# ClinicalRAG Fine-tuning
**Order:** Setup → Data Prep → SFT Training → DPO Training

Before running: set **Runtime → Change runtime type → T4 GPU** (or A100 if on Colab Pro).

## 1. Mount Google Drive
Checkpoints are saved here so they survive session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/clinicalrag-finetune-checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

## 2. Clone repo & install dependencies

In [ ]:
%cd /content
!git clone https://github.com/adyasha95/clinicalrag-finetune.git
%cd clinicalrag-finetune

In [ ]:
!pip install -q -r requirements.txt

## 3. Set API keys
- **WANDB_API_KEY**: get from https://wandb.ai/authorize
- **HF_TOKEN**: get from https://huggingface.co/settings/tokens — needed to download Mistral-7B (gated model)

In [ ]:
import os

os.environ['WANDB_API_KEY'] = 'YOUR_WANDB_API_KEY'   # <-- replace
os.environ['HF_TOKEN']      = 'YOUR_HF_TOKEN'        # <-- replace
os.environ['WANDB_PROJECT'] = 'clinicalrag-finetune'

## 4. Check GPU

In [ ]:
!nvidia-smi

## 5. Prepare SFT dataset
Reads enriched trials data and generates Q&A pairs. Fast — runs on CPU.

In [ ]:
# The script expects enriched trials at ../clinicalrag/data/enriched/trials_enriched.jsonl
# If you have it on Drive, copy it here first:
# !cp "/content/drive/MyDrive/trials_enriched.jsonl" ../clinicalrag/data/enriched/

!python3 -m scripts.prepare_sft_data

## 6. SFT Training (~1-2 hrs on T4, ~30 min on A100)
QLoRA fine-tuning on Mistral-7B-Instruct-v0.3. Saves adapter to `checkpoints/sft-adapter/`.

In [ ]:
!python3 -m scripts.train_sft

In [ ]:
# Save SFT adapter to Drive so it's not lost on session reset
!cp -r checkpoints/sft-adapter "{DRIVE_DIR}/sft-adapter"
print('SFT adapter saved to Drive.')

## 7. DPO Training (~30 min on T4)
Loads SFT adapter, auto-bootstraps preference pairs from the SFT dataset, trains with DPOTrainer.

In [ ]:
# If you're resuming a session, restore the SFT adapter from Drive first:
# import os; os.makedirs('checkpoints/sft-adapter', exist_ok=True)
# !cp -r "{DRIVE_DIR}/sft-adapter" checkpoints/sft-adapter

!python3 -m scripts.train_dpo

In [ ]:
# Save DPO adapter to Drive
!cp -r checkpoints/dpo-adapter "{DRIVE_DIR}/dpo-adapter"
print('DPO adapter saved to Drive.')

## Done
Both adapters are saved to your Google Drive under `clinicalrag-finetune-checkpoints/`.

Next step: run `src/serve/server.py` with vLLM to serve the fine-tuned model.